In [35]:
import random
import numpy as np
import torch

In [36]:
# Without seed → every shuffle is different
# With seed = 3407 → the shuffle order is always the same
# Dataset shuffling
# NumPy-based preprocessing
# Augmentations

# Weight initialization on CPU
# CPU tensor randomness

# GPU randomness
# Dropout
# Attention kernels
# CUDA operations

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


#Do matrix multiplications faster, without breaking training quality.
# This controls how aggressive PyTorch is when using fast math.
# Faster & stable matmul on NVIDIA GPUs
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

max_seq_length = 4096        # Demonstrates long-context + RoPE scaling
dtype = None                # Auto-detect (FP16 on T4, BF16 on A100/L4)
load_in_4bit = True         # Memory-efficient QLoRA-style loading

In [37]:

# GPU sanity check
assert torch.cuda.is_available()

In [38]:
torch.cuda.is_available()

True

In [39]:

# =========================
# Imports
# =========================
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

In [40]:
BASE_MODEL_NAME = "unsloth/gpt-oss-20b"

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=max_seq_length,  # Enables long-context via RoPE scaling
    dtype=dtype,                    # Auto FP16 / BF16
    load_in_4bit=load_in_4bit,      # QLoRA-style memory efficiency
)
    

==((====))==  Unsloth 2026.3.8: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.842 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [10]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,)

In [11]:
tokenizer

LlamaTokenizerFast(name_or_path='unsloth/tinyllama-bnb-4bit', vocab_size=32000, model_max_length=4096, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<unk>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [12]:

# =========================
# Apply LoRA Adapters (PEFT)
# =========================
"""
We attach LoRA adapters so that:
- Only ~1-10% parameters are trained
- Base model weights remain frozen
- Training becomes faster and memory efficient

Target modules:
- q_proj, k_proj, v_proj, o_proj → Attention
- gate_proj, up_proj, down_proj → MLP
"""
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                            # LoRA rank (8/16/32/64 are common)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=32,                   # Usually = r or 2*r
    lora_dropout=0.0,                # Unsloth recommends 0
    bias="none",                     # Best practice for LoRA
    use_gradient_checkpointing=False,# Set True if model >= 7B
    random_state=3407,               # Reproducibility
)

Unsloth 2026.3.8 patched 22 layers with 22 QKV layers, 22 O layers and 22 MLP layers.


In [13]:
model.print_trainable_parameters()

trainable params: 25,231,360 || all params: 1,125,279,744 || trainable%: 2.2422


In [14]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight
base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight
base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight
base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight
base_model.model.model.layers.0.mlp.up_proj.lora_B.default.weight
base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight
base_model.model.model.layers.0.mlp.down_proj.lora_B.default.weight
base_model.model.model.layer

In [15]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable: {trainable}")
print(f"Total: {total}")
print(f"Percent: {100 * trainable / total:.2f}%")

Trainable: 25231360
Total: 640837632
Percent: 3.94%


In [16]:

next(model.parameters()).device

device(type='cuda', index=0)

In [17]:
next(model.parameters()).dtype

torch.bfloat16

In [18]:

from peft import PeftModel
print(isinstance(model, PeftModel))

True


In [19]:
model.peft_config

{'default': LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping={'base_model_class': 'LlamaForCausalLM', 'parent_library': 'transformers.models.llama.modeling_llama', 'unsloth_fixed': True}, peft_version='0.18.1', base_model_name_or_path='unsloth/tinyllama-bnb-4bit', revision=None, inference_mode=False, r=32, target_modules={'up_proj', 'q_proj', 'o_proj', 'down_proj', 'v_proj', 'k_proj', 'gate_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False)

In [20]:
import torch
torch.cuda.memory_summary()

'|===========================================================================|\n|                  PyTorch CUDA memory summary, device ID 0                 |\n|---------------------------------------------------------------------------|\n|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |\n|===========================================================================|\n|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |\n|---------------------------------------------------------------------------|\n| Allocated memory      |    824 MiB |    979 MiB |   1805 MiB |    981 MiB |\n|       from large pool |    701 MiB |    978 MiB |   1679 MiB |    978 MiB |\n|       from small pool |    123 MiB |    123 MiB |    126 MiB |      3 MiB |\n|---------------------------------------------------------------------------|\n| Active memory         |    824 MiB |    979 MiB |   1805 MiB |    981 MiB |\n|       from large pool |    701 MiB |    978 MiB |

'|===========================================================================|\n|                  PyTorch CUDA memory summary, device ID 0                 |\n|---------------------------------------------------------------------------|\n|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |\n|===========================================================================|\n|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |\n|---------------------------------------------------------------------------|\n| Allocated memory      |    824 MiB |    979 MiB |   1805 MiB |    981 MiB |\n|       from large pool |    701 MiB |    978 MiB |   1679 MiB |    978 MiB |\n|       from small pool |    123 MiB |    123 MiB |    126 MiB |      3 MiB |\n|---------------------------------------------------------------------------|\n| Active memory         |    824 MiB |    979 MiB |   1805 MiB |    981 MiB |\n|       from large pool |    701 MiB |    978 MiB |   1679 MiB |    978 MiB |\n|       from small pool |    123 MiB |    123 MiB |    126 MiB |      3 MiB |\n|---------------------------------------------------------------------------|\n| Requested memory      |    824 MiB |    979 MiB |   1805 MiB |    981 MiB |\n|       from large pool |    701 MiB |    978 MiB |   1679 MiB |    978 MiB |\n|       from small pool |    123 MiB |    123 MiB |    126 MiB |      3 MiB |\n|---------------------------------------------------------------------------|\n| GPU reserved memory   |   1126 MiB |   1126 MiB |   1126 MiB |      0 B   |\n|       from large pool |    998 MiB |    998 MiB |    998 MiB |      0 B   |\n|       from small pool |    128 MiB |    128 MiB |    128 MiB |      0 B   |\n|---------------------------------------------------------------------------|\n| Non-releasable memory | 308814 KiB |    871 MiB |    976 MiB | 691396 KiB |\n|       from large pool | 304128 KiB |    871 MiB |    874 MiB | 591781 KiB |\n|       from small pool |   4686 KiB |      4 MiB |    101 MiB |  99615 KiB |\n|---------------------------------------------------------------------------|\n| Allocations           |    1284    |    1284    |    1451    |     167    |\n|       from large pool |     112    |     113    |     114    |       2    |\n|       from small pool |    1172    |    1172    |    1337    |     165    |\n|---------------------------------------------------------------------------|\n| Active allocs         |    1284    |    1284    |    1451    |     167    |\n|       from large pool |     112    |     113    |     114    |       2    |\n|       from small pool |    1172    |    1172    |    1337    |     165    |\n|---------------------------------------------------------------------------|\n| GPU reserved segments |      66    |      66    |      66    |       0    |\n|       from large pool |       2    |       2    |       2    |       0    |\n|       from small pool |      64    |      64    |      64    |       0    |\n|---------------------------------------------------------------------------|\n| Non-releasable allocs |      28    |      29    |      72    |      44    |\n|       from large pool |       3    |       3    |       4    |       1    |\n|       from small pool |      25    |      26    |      68    |      43    |\n|---------------------------------------------------------------------------|\n| Oversize allocations  |       0    |       0    |       0    |       0    |\n|---------------------------------------------------------------------------|\n| Oversize GPU segments |       0    |       0    |       0    |       0    |\n|===========================================================================|\n'

In [21]:

inputs = tokenizer("Hello, how are you?", return_tensors="pt").to("cuda")
with torch.no_grad():
    out = model(**inputs)

print(out.logits.shape)

torch.Size([1, 7, 32000])


In [22]:

model.eval()
print(model.generate(**inputs, max_new_tokens=30))

tensor([[    1, 15043, 29892,   920,   526,   366, 29973,    13,  2855,  3973,
         29901,  1126,   306, 29915, 29885,  2675,   304,   367,  2599,   263,
          3287,   310,   393, 29889,    13,  2855,  3973, 29901,  1126,   306,
         29915, 29885,  2675,   304,   367,  2599,   263]], device='cuda:0')


In [23]:
# =========================
# Dataset Preparation (Alpaca Format)
# =========================
"""
We convert raw Alpaca data into a single 'text' field.
IMPORTANT:
- EOS token is mandatory, otherwise generation may never stop.
"""

EOS_TOKEN = tokenizer.eos_token

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""



def format_data(examples):
    texts = []
    for instruction, input_text, output in zip(
        examples["instruction"],
        examples["input"],
        examples["output"],
    ):
        text = alpaca_prompt.format(
            instruction,
            input_text,
            output
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Load dataset
dataset = load_dataset("yahma/alpaca-cleaned", split="train")

# Shuffle + small subset for fast demo training
dataset = dataset.shuffle(seed=3407).select(range(1500))

# Apply formatting
dataset = dataset.map(
    format_data,
    batched=True,
    remove_columns=dataset.column_names,  # keep only 'text'
)

Map: 100%|██████████| 1500/1500 [00:00<00:00, 122043.34 examples/s]


In [24]:
dataset

Dataset({
    features: ['text'],
    num_rows: 1500
})

In [25]:
dataset["text"][0]

'Below is an instruction that describes a task, paired with an input that provides further context.\nWrite a response that appropriately completes the request.\n\n### Instruction:\nList 5 popular dishes in US.\n\n### Input:\n\n\n### Response:\n1. Hamburger: A classic American dish consisting of a beef patty served on a bun, often topped with cheese and various toppings such as lettuce, tomatoes, onions, and condiments like ketchup and mustard.\n\n2. Macaroni & Cheese: A comforting dish made from elbow macaroni mixed with a creamy cheese sauce and often baked until golden brown and bubbly.\n\n3. Fried Chicken: A southern staple, fried chicken is typically made from pieces of chicken that are coated in a seasoned batter and deep-fried until crispy and golden brown.\n\n4. Pizza: Pizza is a beloved food in the United States, with toppings ranging from classic pepperoni and cheese to more unique combinations like pineapple and ham.\n\n5. Apple Pie: A classic American dessert made from a fla

In [26]:
# psutil = Process & System Utilities

import time, psutil

# Clears unused GPU memory
# Frees cached tensors
# Prevents fake VRAM spikes
# Reset GPU stats

# peak GPU memory used
# This line resets peak counter so:
# You measure only THIS training run
# Not previous runs / notebooks

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [27]:
# your current Python notebook process
# Used to measure CPU RAM consumption
process = psutil.Process()
train_start_time = time.time()
cpu_ram_before = process.memory_info().rss / 1024**3  # GB

In [28]:
process

psutil.Process(pid=2687, name='python', status='running')

In [30]:
train_start_time

1774247384.2467115

In [29]:
cpu_ram_before

2.1874008178710938

In [31]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    packing=True,  # Packs multiple short samples into one sequence
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch size = 8
        num_train_epochs=1,
        learning_rate=2e-5,
        warmup_ratio=0.1,               # Stabilizes early training
        optim="adamw_8bit",             # Memory efficient optimizer
        logging_steps=10,               # Progress visibility
        seed=3407,                      # Reproducibility
        output_dir="outputs",
        report_to="none",               # Disable WandB by default
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=9): 100%|██████████| 1500/1500 [00:00<00:00, 2640.97 examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [32]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,500 | Num Epochs = 1 | Total steps = 188
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 25,231,360 of 1,125,279,744 (2.24% trained)


Step,Training Loss
10,2.258200
20,2.061500
30,1.998400
40,1.485400
50,1.333100
60,1.193500
70,1.146300
80,1.214200
90,1.162300
100,1.151300


TrainOutput(global_step=188, training_loss=1.323393108996939, metrics={'train_runtime': 125.8435, 'train_samples_per_second': 11.92, 'train_steps_per_second': 1.494, 'total_flos': 2239711176781824.0, 'train_loss': 1.323393108996939, 'epoch': 1.0})

In [33]:
train_end_time = time.time()
cpu_ram_after = process.memory_info().rss / 1024**3  # GB

training_time_sec = round(train_end_time - train_start_time, 2)
peak_gpu_vram_gb = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
cpu_ram_used_gb = round(cpu_ram_after - cpu_ram_before, 3)

print("===== UNSLOTH TRAINING STATS =====")
print(f"Training time (sec): {training_time_sec}")
print(f"Peak GPU VRAM (GB): {peak_gpu_vram_gb}")
print(f"CPU RAM used (GB): {cpu_ram_used_gb}")

===== UNSLOTH TRAINING STATS =====
Training time (sec): 543.84
Peak GPU VRAM (GB): 3.104
CPU RAM used (GB): 0.403


In [34]:
# =========================
# Inference (Fast Generation)
# =========================
"""
Unsloth provides 2x faster inference using optimized kernels.
IMPORTANT:
- Always call FastLanguageModel.for_inference(model)
- Disable gradients for inference
"""

FastLanguageModel.for_inference(model)

prompt = alpaca_prompt.format(
    "Continue the Fibonacci sequence",
    "1, 1, 2, 3, 5, 8",
    ""
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        use_cache=True,     # Faster decoding
        do_sample=False     # Deterministic output for demo
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
Continue the Fibonacci sequence

### Input:
1, 1, 2, 3, 5, 8

### Response:
The Fibonacci sequence is a sequence of numbers that are the sum of the previous two numbers. The Fibonacci sequence is named after Leonardo Fibonacci, a 12th-century Italian mathematician. The Fibonacci sequence is a sequence of numbers that are the sum
